This dataset was specifically prepared for evaluating the model after training, with careful attention to avoiding overlap with the training text.
We downloaded the complete works of Shakespeare (pg100.txt) from Project Gutenberg and created input-output pairs based on a line and its following line.
To ensure the evaluation set contains no text the model has seen during training, we skipped the first 15,685 lines of the file,

After skipping, we created pairs only from lines where both input and output contain at least 4 words, shuffled them,
and selected 2,000 pairs for the evaluation.
The dataset was uploaded to Hugging Face at:

https://huggingface.co/datasets/NahlaAboromi/shakespeare-test-pairs


In [ ]:
!pip install bert_score

from transformers import EncoderDecoderModel, BertTokenizer
from datasets import load_dataset
import torch
from bert_score import score
from tqdm import tqdm

def evaluate_model_on_real_data(model_path, dataset, max_samples=200):
    dataset = dataset.select(range(min(max_samples, len(dataset))))
    print(f"✅ Loaded {len(dataset)} real test pairs for evaluation.\n")

    print(f"📦 Loading model and tokenizer from: {model_path}")
    tokenizer = BertTokenizer.from_pretrained(model_path)
    model = EncoderDecoderModel.from_pretrained(model_path)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    print("✅ Model ready on", device)

    predictions = []
    references = []
    displayed_examples = []

    print("🔍 Generating predictions...")
    for pair in tqdm(dataset, desc="Generating"):
        input_text = pair['input']
        ref_output = pair['output']

        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=128).to(device)
        input_length = inputs['input_ids'].shape[1]

        output_ids = model.generate(
            inputs["input_ids"],
            max_length=50 + input_length,
            num_beams=5,
            no_repeat_ngram_size=2,
            early_stopping=True
        )

        full_decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        full_tokenized = tokenizer.tokenize(full_decoded)

        model_generated_tokens = full_tokenized[input_length:]
        trimmed_output = tokenizer.convert_tokens_to_string(model_generated_tokens).strip()

        predictions.append(trimmed_output)
        references.append(ref_output)
        displayed_examples.append((input_text, trimmed_output, ref_output))

    print("\n✅ Finished generating predictions. Now evaluating with BERTScore...")
    P_scores, R_scores, F1_scores = score(predictions, references, lang="en", verbose=True)

    print(f"\n📊 Final Evaluation Result on {len(predictions)} samples:")
    print(f"🌟 BERTScore F1: {F1_scores.mean().item():.4f}")

    print("\n📝 Example Predictions (only model-created part shown):")
    for i in range(min(5, len(displayed_examples))):
        print(f"\n🔹 Example {i+1}")
        print(f"Input:     {displayed_examples[i][0]}")
        print(f"Generated: {displayed_examples[i][1]}")
        print(f"Reference: {displayed_examples[i][2]}")

!wget https://huggingface.co/datasets/NahlaAboromi/shakespeare-test-pairs/resolve/main/test_pairs_real.jsonl -O /content/test_pairs_real.jsonl
import json
from datasets import Dataset

with open("/content/test_pairs_real.jsonl", "r") as f:
    data = [json.loads(line) for line in f]

dataset = Dataset.from_list(data)
# קריאה לפונקציה
evaluate_model_on_real_data(
    model_path="NahlaAboromi/shakespeare-bert-30k-epoch1",
    dataset=dataset,
    max_samples=2000
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 45.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/63 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/32 [00:00<?, ?it/s]

done in 13.42 seconds, 149.02 sentences/sec

📊 Final Evaluation Result on 2000 samples:
🌟 BERTScore F1: 0.8019

📝 Example Predictions (only model-created part shown):

🔹 Example 1
Input:     I have in quick determination
Generated: determination ; have of quick determined determination determined on quick triumph determination , that thou dost ’ st not to have the
Reference: Thus set it down: he shall with speed to England

🔹 Example 2
Input:     Her chamber is aloft, far from the ground,
Generated: ##ving , well , , thine , with the earth , mine own love ’ s sun ,
Reference: And built so shelving that one cannot climb it

🔹 Example 3
Input:     He has discover’d my design, and I
Generated: t , but i am i , that i ’ s thine own me . countess .
Reference: Remain a pinch’d thing; yea, a very trick

🔹 Example 4
Input:     In my just censure, in my true opinion!
Generated: judgment ! i true sense ! what true ears , that my love ’ s own
Reference: Alack, for lesser knowledge! How accurs’d

